# 11 — Imbalance / Synthetic-Data Benchmark with Rolling Time-Series CV

**synthetic data generation for the multiclass problem**.

**Main design choice**
Because the dataset contains an important **categorical spatial feature** (`HOOD_158_CODE`), the most defensible synthetic benchmark here is **SMOTENC**.

**Compared strategies**
- **Weight only** (no resampling, just class-balanced learning)
- **RandomOverSampler** (simple duplication baseline)
- **SMOTENC** (category-aware synthetic oversampling)

**Important safety rule**
All resampling is done **inside each training fold only**. We do **not** resample before the time split, because that would leak future information.

**Practical note**
Full-data SMOTE on this dataset is very heavy. For speed and stability, this notebook:
- keeps **all class 1 and class 2** rows inside each fold
- randomly downsamples a fraction of class 0 inside each fold
- then applies the resampler

That makes this notebook practical for capstone use while still being methodologically correct.


In [23]:

from __future__ import annotations

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    average_precision_score,
    mean_squared_error,
)

warnings.filterwarnings("ignore")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SUPERVISED_PATH = DATA_DIR / "supervised_hood_3h_multiclass.csv"
DEV_END = pd.Timestamp("2025-06-30 23:59:59")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTENC

MIN_TRAIN_DAYS = 365
VAL_DAYS = 90
STEP_DAYS = 90
MAX_FOLDS = 4
MAJORITY_FRAC = 0.20   # keep 20% of class 0 inside each fold for speed

USE_FEATURE_FILE_IF_AVAILABLE = True
FEATURE_FILE_CANDIDATES = [
    MODEL_DIR / "catboost_selected_features.txt",
    MODEL_DIR / "best_ml_catboost_features.txt",
]

OUT_CV_SUMMARY = MODEL_DIR / "sampling_cv_summary.csv"
OUT_TEST = MODEL_DIR / "sampling_final_summary.csv"


In [19]:

df = pd.read_csv(SUPERVISED_PATH, low_memory=False)
df["time_3h"] = pd.to_datetime(df["time_3h"], errors="coerce")
df["HOOD_158_CODE"] = df["HOOD_158_CODE"].astype(str).str.zfill(3)
df["y_class"] = pd.to_numeric(df["y_class"], errors="coerce")
df = df.dropna(subset=["time_3h", "y_class"]).copy()
df["y_class"] = df["y_class"].astype("int8")
df = df.sort_values(["time_3h", "HOOD_158_CODE"]).reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in ["time_3h", "y_class", "y_count_next"]]
if USE_FEATURE_FILE_IF_AVAILABLE:
    for fp in FEATURE_FILE_CANDIDATES:
        if fp.exists():
            with open(fp, "r", encoding="utf-8") as f:
                selected = [line.strip() for line in f if line.strip()]
            feature_cols = [c for c in selected if c in feature_cols]
            print("Using feature file:", fp)
            break

dev_mask = df["time_3h"] <= DEV_END
test_mask = df["time_3h"] > DEV_END

X_dev = df.loc[dev_mask, feature_cols].reset_index(drop=True)
y_dev = df.loc[dev_mask, "y_class"].reset_index(drop=True)
t_dev = df.loc[dev_mask, "time_3h"].reset_index(drop=True)

X_test = df.loc[test_mask, feature_cols].reset_index(drop=True)
y_test = df.loc[test_mask, "y_class"].reset_index(drop=True)

print("Development shape:", X_dev.shape)
print("Test shape:", X_test.shape)
print("Feature count:", len(feature_cols))
print(y_dev.value_counts(normalize=True).sort_index().round(4))


Using feature file: ../models/best_ml_catboost_features.txt
Development shape: (1151504, 47)
Test shape: (232418, 47)
Feature count: 47
y_class
0    0.8927
1    0.0930
2    0.0143
Name: proportion, dtype: float64


In [20]:
def metric_row(y_true, pred, proba):
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(pred)
    proba_arr = np.asarray(proba)

    rec = recall_score(
        y_true_arr, y_pred_arr,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    )
    macro_rec = float(np.mean(rec))

    prec2 = precision_score(
        (y_true_arr == 2).astype(int),
        (y_pred_arr == 2).astype(int),
        zero_division=0
    )
    pred2_rate = float((y_pred_arr == 2).mean())

    # AP / MAP
    ap_vals = []
    class_support = []
    ap_dict = {}

    for cls in [0, 1, 2]:
        y_true_bin = (y_true_arr == cls).astype(int)
        support = int(y_true_bin.sum())
        class_support.append(support)

        if support == 0:
            ap = np.nan
        else:
            ap = float(average_precision_score(y_true_bin, proba_arr[:, cls]))

        ap_dict[f"ap_class{cls}"] = ap
        if not np.isnan(ap):
            ap_vals.append(ap)

    macro_map = float(np.mean(ap_vals)) if len(ap_vals) else np.nan

    valid_pairs = [
        (ap_dict[f"ap_class{cls}"], class_support[cls])
        for cls in [0, 1, 2]
        if not np.isnan(ap_dict[f"ap_class{cls}"])
    ]
    if len(valid_pairs):
        weighted_map = float(
            np.average(
                [x[0] for x in valid_pairs],
                weights=[x[1] for x in valid_pairs]
            )
        )
    else:
        weighted_map = np.nan

    # RMSE
    label_rmse = float(np.sqrt(mean_squared_error(y_true_arr, y_pred_arr)))

    y_onehot = np.zeros((len(y_true_arr), 3), dtype=float)
    y_onehot[np.arange(len(y_true_arr)), y_true_arr.astype(int)] = 1.0
    prob_rmse_macro = float(
        np.mean([
            np.sqrt(mean_squared_error(y_onehot[:, c], proba_arr[:, c]))
            for c in range(3)
        ])
    )

    row = {
        "macro_recall": macro_rec,
        "recall_0": float(rec[0]),
        "recall_1": float(rec[1]),
        "recall_2": float(rec[2]),
        "precision_2": float(prec2),
        "pred2_rate": pred2_rate,
        "macro_map": macro_map,
        "weighted_map": weighted_map,
        "label_rmse": label_rmse,
        "prob_rmse_macro": prob_rmse_macro,
    }
    row.update(ap_dict)
    return row

In [21]:

# ============================================================
# CV helpers: folds, preprocessing, sampling, model
# ============================================================
def make_rolling_time_folds(unique_times, min_train_days=365, val_days=90, step_days=90, max_folds=4):
    times = pd.Series(pd.to_datetime(pd.Index(unique_times).unique())).sort_values().reset_index(drop=True)
    freq = pd.Timedelta(hours=3)
    min_train_periods = int(pd.Timedelta(days=min_train_days) / freq)
    val_periods = int(pd.Timedelta(days=val_days) / freq)
    step_periods = int(pd.Timedelta(days=step_days) / freq)

    folds = []
    end = min_train_periods + val_periods
    while end <= len(times):
        train_end_idx = end - val_periods
        folds.append({
            "train_start": times.iloc[0],
            "train_end": times.iloc[train_end_idx - 1],
            "val_start": times.iloc[train_end_idx],
            "val_end": times.iloc[end - 1],
        })
        end += step_periods

    if max_folds is not None:
        folds = folds[-max_folds:]
    return folds


cat_cols = [c for c in ["HOOD_158_CODE"] if c in X_dev.columns]
num_cols = [c for c in X_dev.columns if c not in cat_cols]

folds = make_rolling_time_folds(
    t_dev.unique(),
    min_train_days=MIN_TRAIN_DAYS,
    val_days=VAL_DAYS,
    step_days=STEP_DAYS,
    max_folds=MAX_FOLDS,
)
print("Folds:", len(folds))
for i, f in enumerate(folds, start=1):
    print(f"Fold {i}: train {f['train_start']} -> {f['train_end']} | val {f['val_start']} -> {f['val_end']}")


def fit_preprocessor(Xtr):
    num_imputer = SimpleImputer(strategy="median")
    cat_imputer = SimpleImputer(strategy="most_frequent")
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    Xtr_num = num_imputer.fit_transform(Xtr[num_cols]) if num_cols else np.empty((len(Xtr), 0))
    if cat_cols:
        Xtr_cat_raw = cat_imputer.fit_transform(Xtr[cat_cols])
        Xtr_cat = encoder.fit_transform(Xtr_cat_raw)
    else:
        Xtr_cat = np.empty((len(Xtr), 0))

    Xtr_out = np.hstack([Xtr_num, Xtr_cat]).astype(np.float32)
    state = {
        "num_imputer": num_imputer,
        "cat_imputer": cat_imputer,
        "encoder": encoder,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "cat_feature_idx": list(range(len(num_cols), len(num_cols) + len(cat_cols))),
    }
    return Xtr_out, state


def transform_preprocessor(X, state):
    X_num = state["num_imputer"].transform(X[state["num_cols"]]) if state["num_cols"] else np.empty((len(X), 0))
    if state["cat_cols"]:
        X_cat_raw = state["cat_imputer"].transform(X[state["cat_cols"]])
        X_cat = state["encoder"].transform(X_cat_raw)
    else:
        X_cat = np.empty((len(X), 0))
    return np.hstack([X_num, X_cat]).astype(np.float32)


def make_fold_training_subset(X, y, majority_frac=MAJORITY_FRAC, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)

    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    idx2 = np.where(y == 2)[0]

    take0 = max(1, int(len(idx0) * majority_frac))
    keep0 = rng.choice(idx0, size=take0, replace=False) if len(idx0) else np.array([], dtype=int)
    keep = np.concatenate([keep0, idx1, idx2])
    rng.shuffle(keep)
    return X[keep], y[keep]


def build_sampling_strategy(y):
    counts = pd.Series(y).value_counts().sort_index()
    n0 = int(counts.get(0, 0))
    n1 = int(counts.get(1, 0))
    n2 = int(counts.get(2, 0))
    target1 = max(n1, int(0.50 * n0))
    target2 = max(n2, int(0.15 * n0))
    strategy = {}
    if n1 < target1:
        strategy[1] = target1
    if n2 < target2:
        strategy[2] = target2
    return strategy


def build_lgbm(class_weight=None, seed=RANDOM_SEED):
    return LGBMClassifier(
        objective="multiclass",
        num_class=3,
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=100,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=seed,
        class_weight=class_weight,
        n_jobs=-1,
    )
def fit_one_strategy(strategy_name, Xtr_df, ytr, Xva_df, yva, fold_id):
    # Preprocess
    Xtr_all, prep_state = fit_preprocessor(Xtr_df)
    Xva_all = transform_preprocessor(Xva_df, prep_state)

    # Optional train-size reduction for speed
    Xtr_small, ytr_small = make_fold_training_subset(
        Xtr_all, ytr,
        majority_frac=MAJORITY_FRAC,
        seed=RANDOM_SEED + fold_id
    )

    # Validation data is NEVER resampled
    Xva_use = Xva_all

    # Default training set
    Xtr_use, ytr_use = Xtr_small, np.asarray(ytr_small)
    class_weight = None

    if strategy_name == "weight_only":
        class_weight = "balanced"

    elif strategy_name == "random_over":
        ros = RandomOverSampler(
            sampling_strategy=build_sampling_strategy(ytr_small),
            random_state=RANDOM_SEED + fold_id
        )
        Xtr_use, ytr_use = ros.fit_resample(Xtr_small, ytr_small)

    elif strategy_name == "smotenc":
        if len(prep_state["cat_feature_idx"]) == 0:
            ros = RandomOverSampler(
                sampling_strategy=build_sampling_strategy(ytr_small),
                random_state=RANDOM_SEED + fold_id
            )
            Xtr_use, ytr_use = ros.fit_resample(Xtr_small, ytr_small)
        else:
            sm = SMOTENC(
                categorical_features=prep_state["cat_feature_idx"],
                sampling_strategy=build_sampling_strategy(ytr_small),
                random_state=RANDOM_SEED + fold_id,
                k_neighbors=3
            )
            Xtr_use, ytr_use = sm.fit_resample(Xtr_small, ytr_small)

    else:
        raise ValueError(f"Unknown strategy: {strategy_name}")

    # Model
    model = build_lgbm(class_weight=class_weight, seed=RANDOM_SEED + fold_id)
    model.fit(Xtr_use, ytr_use)

    # Evaluate on untouched validation set
    proba = model.predict_proba(Xva_use)
    pred = np.argmax(proba, axis=1).astype(int)

    row = metric_row(yva, pred, proba)
    return row


rows = []
# ============================================================
# Build rolling time-series CV folds
# ============================================================
import numpy as np
import pandas as pd

# make sure development timestamps are datetime
t_dev = pd.to_datetime(t_dev)

# sorted unique timestamps in development set
unique_times = np.array(sorted(pd.Series(t_dev).dropna().unique()))
print("Unique dev timestamps:", len(unique_times))

N_SPLITS = 3
MIN_TRAIN_FRAC = 0.55
VAL_FRAC = 0.15

n_times = len(unique_times)
val_size = max(1, int(n_times * VAL_FRAC))
start_train_end = max(2, int(n_times * MIN_TRAIN_FRAC))

folds = []
for i in range(N_SPLITS):
    train_end_idx = start_train_end + i * val_size
    val_start_idx = train_end_idx
    val_end_idx = min(train_end_idx + val_size - 1, n_times - 1)

    if val_start_idx >= n_times or val_start_idx > val_end_idx:
        break

    folds.append({
        "train_start": unique_times[0],
        "train_end": unique_times[train_end_idx - 1],
        "val_start": unique_times[val_start_idx],
        "val_end": unique_times[val_end_idx],
    })

print("Number of folds:", len(folds))
display(pd.DataFrame(folds))
for fold_id, f in enumerate(folds, start=1):
    tr_mask = (t_dev >= f["train_start"]) & (t_dev <= f["train_end"])
    va_mask = (t_dev >= f["val_start"]) & (t_dev <= f["val_end"])

    Xtr, ytr = X_dev.loc[tr_mask].copy(), y_dev.loc[tr_mask].copy()
    Xva, yva = X_dev.loc[va_mask].copy(), y_dev.loc[va_mask].copy()

    for strategy in ["weight_only", "random_over", "smotenc"]:
        row = fit_one_strategy(strategy, Xtr, ytr, Xva, yva, fold_id=fold_id)
        row.update({
            "strategy": strategy,
            "fold": fold_id,
            "train_start": f["train_start"],
            "train_end": f["train_end"],
            "val_start": f["val_start"],
            "val_end": f["val_end"],
            "n_train_full": int(len(Xtr)),
            "n_val": int(len(Xva)),
        })
        rows.append(row)
        print(f"Done: {strategy} | fold {fold_id}/{len(folds)} | macro={row['macro_recall']:.4f} | prec2={row['precision_2']:.4f} | ap2={row.get('ap_class2', np.nan):.4f}")

cv_rows = pd.DataFrame(rows)
cv_rows["selection_score"] = cv_rows["macro_recall"] + 0.20 * cv_rows["precision_2"] + 0.10 * cv_rows["ap_class2"].fillna(0)

cv_summary = (
    cv_rows.groupby("strategy", as_index=False)
    .agg(
        folds=("fold", "count"),
        selection_score_mean=("selection_score", "mean"),
        macro_recall_mean=("macro_recall", "mean"),
        macro_recall_std=("macro_recall", "std"),
        recall_1_mean=("recall_1", "mean"),
        recall_2_mean=("recall_2", "mean"),
        precision_2_mean=("precision_2", "mean"),
        pred2_rate_mean=("pred2_rate", "mean"),
        ap_class2_mean=("ap_class2", "mean"),
        macro_map_mean=("macro_map", "mean"),
        weighted_map_mean=("weighted_map", "mean"),
        label_rmse_mean=("label_rmse", "mean"),
        prob_rmse_macro_mean=("prob_rmse_macro", "mean"),
    )
    .sort_values(["selection_score_mean", "macro_recall_mean", "ap_class2_mean"], ascending=False)
    .reset_index(drop=True)
)

cv_summary_to_save = cv_summary[[c for c in ["strategy","macro_recall_mean","recall_2_mean","precision_2_mean","ap_class2_mean","macro_map_mean","prob_rmse_macro_mean"] if c in cv_summary.columns]].copy()
cv_summary_to_save.to_csv(OUT_CV_SUMMARY, index=False)
print("Saved:", OUT_CV_SUMMARY)
display(cv_summary)


Folds: 4
Fold 1: train 2023-01-02 00:00:00 -> 2024-06-29 21:00:00 | val 2024-06-30 00:00:00 -> 2024-09-27 21:00:00
Fold 2: train 2023-01-02 00:00:00 -> 2024-09-27 21:00:00 | val 2024-09-28 00:00:00 -> 2024-12-26 21:00:00
Fold 3: train 2023-01-02 00:00:00 -> 2024-12-26 21:00:00 | val 2024-12-27 00:00:00 -> 2025-03-26 21:00:00
Fold 4: train 2023-01-02 00:00:00 -> 2025-03-26 21:00:00 | val 2025-03-27 00:00:00 -> 2025-06-24 21:00:00
Unique dev timestamps: 7288
Number of folds: 3


,train_start,train_end,val_start,val_end
0,2023-01-02,2024-05-16 21:00:00,2024-05-17 00:00:00,2024-09-30 12:00:00
1,2023-01-02,2024-09-30 12:00:00,2024-09-30 15:00:00,2025-02-14 03:00:00
2,2023-01-02,2025-02-14 03:00:00,2025-02-14 06:00:00,2025-06-30 18:00:00


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011206 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2573
[LightGBM] [Info] Number of data points in the train set: 180469, number of used features: 47
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Done: weight_only | fold 1/3 | macro=0.5343 | prec2=0.0826 | ap2=0.1154
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010338 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2573
[LightGBM] [Info] Number of data points in the train set: 188662, number of used features: 47
[LightGBM] [Info] Start training from score -0.510819
[Lig

,strategy,folds,selection_score_mean,macro_recall_mean,macro_recall_std,recall_1_mean,recall_2_mean,precision_2_mean,pred2_rate_mean,ap_class2_mean,macro_map_mean,weighted_map_mean,label_rmse_mean,prob_rmse_macro_mean
0,weight_only,3,0.558768,0.532019,0.002400,0.473139,0.476680,0.079808,0.088389,0.107874,0.403927,0.870386,0.745893,0.377464
1,smotenc,3,0.505237,0.427012,0.002825,0.383631,0.025246,0.336253,0.001162,0.109747,0.422941,0.875666,0.449591,0.287110
2,random_over,3,0.496275,0.439969,0.003645,0.371779,0.076621,0.226517,0.005103,0.110026,0.420677,0.875016,0.456433,0.291102


In [24]:

best_strategy = cv_summary.iloc[0]["strategy"]
print("Best strategy from CV:", best_strategy)

# Refit best strategy on the full development period using the same safe recipe as CV
Xdev_all, prep_state = fit_preprocessor(X_dev)
Xtest_all = transform_preprocessor(X_test, prep_state)
Xdev_small, ydev_small = make_fold_training_subset(Xdev_all, y_dev, majority_frac=MAJORITY_FRAC, seed=RANDOM_SEED)
sampling_strategy = build_sampling_strategy(ydev_small)

if best_strategy == "weight_only":
    best_model = build_lgbm(class_weight="balanced", seed=RANDOM_SEED)
    sw = compute_sample_weight(class_weight="balanced", y=ydev_small)
    best_model.fit(Xdev_small, ydev_small, sample_weight=sw)
elif best_strategy == "random_over":
    sampler = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=RANDOM_SEED)
    Xfit, yfit = sampler.fit_resample(Xdev_small, ydev_small)
    best_model = build_lgbm(class_weight=None, seed=RANDOM_SEED)
    best_model.fit(Xfit, yfit)
elif best_strategy == "smotenc":
    sampler = SMOTENC(
        categorical_features=prep_state["cat_feature_idx"],
        sampling_strategy=sampling_strategy,
        k_neighbors=5,
        random_state=RANDOM_SEED,
    )
    Xfit, yfit = sampler.fit_resample(Xdev_small, ydev_small)
    best_model = build_lgbm(class_weight=None, seed=RANDOM_SEED)
    best_model.fit(Xfit, yfit)
else:
    raise ValueError(best_strategy)

test_proba = best_model.predict_proba(Xtest_all)
test_pred = np.argmax(test_proba, axis=1)
test_metrics = print_metrics(y_test, test_pred, proba=test_proba, label=f"{best_strategy} - Final Test")

test_df = pd.DataFrame([test_metrics])
if "model" not in test_df.columns:
    test_df.insert(0, "model", best_strategy)
test_df.to_csv(OUT_TEST, index=False)

print("Saved:", OUT_TEST)
display(test_df)


Best strategy from CV: weight_only
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.149090 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2771
[LightGBM] [Info] Number of data points in the train set: 329124, number of used features: 47
[LightGBM] [Info] Start training from score -2.736892
[LightGBM] [Info] Start training from score -2.084826
[LightGBM] [Info] Start training from score -0.209611


,model,macro_recall,recall_0,recall_1,recall_2,precision_2,ap_class0,ap_class1,ap_class2,macro_map,prob_rmse_macro
0,weight_only - Final Test,0.467248,0.373485,0.07839,0.949868,0.024687,0.961362,0.080631,0.112772,0.384922,0.515044


Saved: ../models/sampling_final_summary.csv


,model,macro_recall,recall_0,recall_1,recall_2,precision_2,ap_class0,ap_class1,ap_class2,macro_map,prob_rmse_macro
0,weight_only - Final Test,0.467248,0.373485,0.07839,0.949868,0.024687,0.961362,0.080631,0.112772,0.384922,0.515044
